# Phase 23 — Relational Substep Diagnosis (Phase 15)
## NeuroForge Experimental Research

Research question: can a minimal, controlled increase or modification of relational computation convert the existing decodable-but-not-task-usable relational signal into stronger relational prediction? Distinguish: (1) message-passing depth, (2) neighbourhood aggregation, (3) relational update capacity, (4) fixed topology, (5) branch interaction. No assumption is made before experimentation.

## 1. Environment verification

In [1]:
import platform, torch, neuroforge
from pathlib import Path
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'NeuroForge: {neuroforge.__file__}')

Python: 3.13.14
PyTorch: 2.13.0+cpu
NeuroForge: C:\Projects\NN\src\neuroforge\__init__.py


## 2. Phase 14 baseline (frozen reference)

In [2]:
import json
p14_path = Path('../results/metrics/phase14_readout_diagnosis/summary.json')
p14 = json.loads(p14_path.read_text(encoding='utf-8'))
base14 = p14['baseline_perf_mean']
print('Phase 14 JointCo baseline (original head):')
print(f"  R={base14['R']*100:.1f}%  RC={base14['RC']*100:.1f}%  FRC={base14['FRC']*100:.1f}%  mixed={base14['mixed_mean']*100:.1f}%")
print(f"  R-signal probe={p14['r_probe_mean']*100:.1f}%  rel_only branch={p14['branch_readout_results_mean']['rel_only']['R']*100:.1f}%")

Phase 14 JointCo baseline (original head):
  R=62.2%  RC=53.3%  FRC=80.3%  mixed=65.0%
  R-signal probe=77.2%  rel_only branch=48.6%


## 3. Exact current relational computation (15A audit — from the implementation itself)

In [3]:
from neuroforge.blocks.joint_co_relational import JointCoRelationalBlock
import json as _json
audit = JointCoRelationalBlock().describe_relational_computation()
print(_json.dumps(audit, indent=1))
print()
print('WHAT: one ring message-passing round; msgs = A_norm @ Linear(h); rel = tanh(Linear([h;msgs]))')
print('WHEN: inside JointCo after the encoder, parallel to feature/context branches, before fusion+residual')
print('HOW: neighbour info enters only via the single averaged A @ W(h) step')

{
 "input_shape": "[B, S, H]",
 "graph_construction": "fixed ring: each node connects to self + left + right neighbour",
 "topology": "ring (circulant, degree 3 including self-loop)",
 "message_passing_rounds": 1,
 "neighbourhood_aggregation": "mean",
 "aggregation_semantics": "row-normalised adjacency: average of self + 2 neighbours",
 "self_feature_handling": "self features concatenated with neighbourhood messages ([h ; msgs])",
 "update_transformation": "tanh(Linear(2H -> H)([h ; msgs]))",
 "activation": "tanh",
 "normalisation": "none inside relational substep (LayerNorm only in final classifier head)",
 "residual_connection": "block-level residual: out = state + scaled_sum + gated_fusion",
 "output_shape": "[B, S, H]",
 "relational_param_count": 1776,
 "relational_params_per_round": {
  "message": 600,
  "update": 1176
 },
 "analytical_flops_per_round": 1728,
 "when_applied": "inside JointCo block, after the shared encoder linear, in parallel with the feature and contextual branch

## 4. Depth ablation (15B: depth 1 = baseline, 2, 3; everything else fixed)

In [4]:
import json
p15 = json.loads(Path('../results/metrics/phase15_relational_diagnosis/summary.json').read_text(encoding='utf-8'))
depth = p15['depth_results_mean']
for cond in ('baseline', 'depth2', 'depth3'):
    d = depth.get(cond, {})
    print(f"{cond:>9}: " + '  '.join(f"{f}={d.get(f, 0)*100:.1f}%" for f in ('F','R','C','FR','RC','FC','FRC')))
print()
for r in p15['per_seed_results']:
    b = r['baseline_eval']['perf']['R']
    print(f"seed {r['seed']}: baseline R={b*100:.1f}%  depth2 R={r['depth2_eval']['perf']['R']*100:.1f}%  depth3 R={r['depth3_eval']['perf']['R']*100:.1f}%")

 baseline: F=100.0%  R=57.8%  C=98.9%  FR=76.9%  RC=53.9%  FC=49.4%  FRC=79.4%
   depth2: F=100.0%  R=57.2%  C=99.4%  FR=76.7%  RC=53.3%  FC=49.7%  FRC=80.3%
   depth3: F=100.0%  R=62.5%  C=99.2%  FR=76.7%  RC=53.6%  FC=50.0%  FRC=80.0%

seed 11: baseline R=50.0%  depth2 R=43.3%  depth3 R=60.0%
seed 23: baseline R=60.0%  depth2 R=61.7%  depth3 R=63.3%
seed 37: baseline R=63.3%  depth2 R=66.7%  depth3 R=64.2%


## 5. Aggregation diagnosis (15C — runs only if depth does not fully explain; gate recorded in summary)

In [5]:
ag = p15.get('aggregation_results_mean', {})
print('gate:', p15['gate_info'].get('aggregation_run'), '|', '; '.join(p15['gate_info'].get('reasons', [])))
for cond, d in ag.items():
    print(f"{cond:>9}: " + '  '.join(f"{f}={d.get(f, 0)*100:.1f}%" for f in ('R','RC','FRC')))
if not ag:
    print('NOT TESTED (gated off)')

gate: True | Depth does not fully explain (best depth3: R gain +4.7pp, gap closes -2.5pp): running aggregation per §7.; Aggregation inadequate (best agg_max: +4.2pp): running capacity per §8.
  agg_sum: R=60.0%  RC=53.6%  FRC=79.4%
  agg_max: R=61.9%  RC=53.9%  FRC=80.3%


## 6. Update-capacity diagnosis (15D — runs only if depth/aggregation do not explain)

In [6]:
cp = p15.get('capacity_results_mean', {})
print('gate:', p15['aggregates']['capacity'].get('tested'))
for cond, d in cp.items():
    print(f"{cond:>22}: " + '  '.join(f"{f}={d.get(f, 0)*100:.1f}%" for f in ('R','RC','FRC')))
if not cp:
    print('NOT TESTED (gated off)')

gate: True
               cap_mlp: R=61.7%  RC=53.6%  FRC=79.7%
  cap_control_featwide: R=52.5%  RC=53.6%  FRC=80.3%


## 7. Topology controls (15E — normal vs destructive correspondence permutation)

In [7]:
topo = p15['aggregates']['topology']
print(f"baseline destruction drop on R: {topo['baseline_drop_R']*100:+.1f}pp")
print(f"candidate ({topo['candidate']}) destruction drop on R: {topo['candidate_drop_R']*100:+.1f}pp")
print(f"dependence delta: {topo['candidate_drop_minus_baseline_drop_R']*100:+.1f}pp (drop alone is not reasoning; needs performance)")

baseline destruction drop on R: +5.3pp
candidate (depth3) destruction drop on R: +3.6pp
dependence delta: -1.7pp (drop alone is not reasoning; needs performance)


## 8. Causal controls (15F — marker ablation, token permutation, relational destruction)

In [8]:
import csv
rows = list(csv.DictReader(open('../results/metrics/phase15_relational_diagnosis/causal_controls.csv')))
cand = p15['candidate_cond']
for row in rows:
    if row['seed'] == str(p15['per_seed_results'][0]['seed']) and row['condition'] == cand:
        print(f"{row['control']:>11} {row['variant']:>22}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")

destruction               original: R=60.0% RC=51.7% FRC=80.0%
destruction    relational_permuted: R=56.7% RC=50.8% FRC=80.0%
     marker               original: R=60.0% RC=51.7% FRC=80.0%
     marker     marker_neutralised: R=45.8% RC=51.7% FRC=48.3%
      token               original: R=60.0% RC=51.7% FRC=80.0%
      token         token_permuted: R=52.5% RC=50.8% FRC=80.0%


## 9. Branch ablations (15G — F/R/C only and pairs for the best candidate)

In [9]:
cand = p15['candidate_cond']
br = p15['per_seed_results'][0][cand + '_eval']['branch7']
for combo, perf in br.items():
    print(f"{combo:>6}: " + '  '.join(f"{f}={perf.get(f, 0)*100:.1f}%" for f in ('R','RC','FRC')))
print()
print('Required check: if R improves but RC/FRC do not -> isolated capability, no compositional transfer.')

     F: R=55.8%  RC=51.7%  FRC=72.5%
     R: R=53.3%  RC=49.2%  FRC=72.5%
     C: R=47.5%  RC=51.7%  FRC=80.0%
   F+R: R=61.7%  RC=50.0%  FRC=70.8%
   R+C: R=54.2%  RC=51.7%  FRC=80.0%
   F+C: R=55.8%  RC=51.7%  FRC=80.0%
 F+R+C: R=60.0%  RC=51.7%  FRC=80.0%

Required check: if R improves but RC/FRC do not -> isolated capability, no compositional transfer.


## 10. Representation probe reassessment (15H — baseline vs candidate)

In [10]:
g = p15['aggregates']['gap']
print(f"baseline:  probe R={g['baseline_probe_R_mean']*100:.1f}%  final R={g['baseline_final_R_mean']*100:.1f}%  gap={g['baseline_gap_mean']*100:.1f}pp")
print(f"candidate: probe R={g['candidate_probe_R_mean']*100:.1f}%  final R={g['candidate_final_R_mean']*100:.1f}%  gap={g['candidate_gap_mean']*100:.1f}pp")
print(f"gap closes {g['gap_close_mean']*100:+.1f}pp")

baseline:  probe R=76.9%  final R=57.8%  gap=19.2pp
candidate: probe R=84.2%  final R=62.5%  gap=21.7pp
gap closes -2.5pp


## 11. Standalone Graph comparison (15I — is JointCo's relational path weaker than Graph?)

In [11]:
import csv
rows = list(csv.DictReader(open('../results/metrics/phase15_relational_diagnosis/graph_comparison.csv')))
seen = set()
for row in rows:
    key = (row['seed'], row['condition'])
    if row['seed'] == str(p15['per_seed_results'][0]['seed']) and key not in seen:
        seen.add(key)
        print(f"{row['condition']:>18}: R={float(row['R'])*100:.1f}% drop={float(row['destruction_drop_R'])*100:+.1f}pp probe={float(row['probe_R'])*100:.1f}% params={row['params']}")

  graph_standalone: R=90.8% drop=+47.5pp probe=95.8% params=3866
  jointco_baseline: R=50.0% drop=+1.7pp probe=81.7% params=6464


## 12. Compute and latency (FLOPs are not latency — both reported)

In [12]:
import csv
for name in ('compute.csv', 'latency.csv'):
    print(f'--- {name} (seed {p15["per_seed_results"][0]["seed"]}) ---')
    for row in csv.DictReader(open(f'../results/metrics/phase15_relational_diagnosis/{name}')):
        if row['seed'] == str(p15['per_seed_results'][0]['seed']):
            print(' ', {k: v for k, v in row.items() if k != 'seed'})

--- compute.csv (seed 11) ---


  {'condition': 'baseline', 'params': '6464', 'rel_params': '1776', 'activation_bytes': '69120'}
  {'condition': 'depth2', 'params': '8240', 'rel_params': '3552', 'activation_bytes': '138240'}
  {'condition': 'depth3', 'params': '10016', 'rel_params': '5328', 'activation_bytes': '207360'}
  {'condition': 'agg_sum', 'params': '6464', 'rel_params': '1776', 'activation_bytes': '69120'}
  {'condition': 'agg_max', 'params': '6464', 'rel_params': '1776', 'activation_bytes': '69120'}
  {'condition': 'cap_mlp', 'params': '7064', 'rel_params': '2376', 'activation_bytes': '69120'}
  {'condition': 'cap_control_featwide', 'params': '7052', 'rel_params': '1776', 'activation_bytes': '69120'}
--- latency.csv (seed 11) ---
  {'condition': 'baseline', 'latency_us': '64.08225000389696', 'throughput_per_s': '15604.945206187176'}
  {'condition': 'depth2', 'latency_us': '69.85275000261026', 'throughput_per_s': '14315.828653311886'}
  {'condition': 'depth3', 'latency_us': '100.76583334011957', 'throughput_p

## 13. Seed aggregation (mean ± SD, never best-seed)

In [13]:
import csv
for row in csv.DictReader(open('../results/metrics/phase15_relational_diagnosis/seed_results.csv')):
    print(' ', {k: (f'{float(v)*100:.1f}%' if k != 'seed' else v) for k, v in row.items()})
print('stability (R seed-SD):', {k: {c: f'{v*100:.1f}pp' for c, v in d.items()} for k, d in p15['aggregates']['seed_stability'].items()})

  {'seed': '11', 'baseline_R': '50.0%', 'baseline_RC': '50.8%', 'baseline_FRC': '80.0%', 'depth2_R': '43.3%', 'depth2_RC': '51.7%', 'depth2_FRC': '80.0%', 'depth3_R': '60.0%', 'depth3_RC': '51.7%', 'depth3_FRC': '80.0%', 'agg_sum_R': '55.8%', 'agg_sum_RC': '50.8%', 'agg_sum_FRC': '80.0%', 'agg_max_R': '57.5%', 'agg_max_RC': '51.7%', 'agg_max_FRC': '80.0%', 'cap_mlp_R': '57.5%', 'cap_mlp_RC': '51.7%', 'cap_mlp_FRC': '80.0%', 'cap_control_featwide_R': '45.0%', 'cap_control_featwide_RC': '51.7%', 'cap_control_featwide_FRC': '80.0%'}
  {'seed': '23', 'baseline_R': '60.0%', 'baseline_RC': '51.7%', 'baseline_FRC': '83.3%', 'depth2_R': '61.7%', 'depth2_RC': '50.8%', 'depth2_FRC': '83.3%', 'depth3_R': '63.3%', 'depth3_RC': '50.8%', 'depth3_FRC': '83.3%', 'agg_sum_R': '62.5%', 'agg_sum_RC': '51.7%', 'agg_sum_FRC': '81.7%', 'agg_max_R': '60.8%', 'agg_max_RC': '50.8%', 'agg_max_FRC': '82.5%', 'cap_mlp_R': '62.5%', 'cap_mlp_RC': '51.7%', 'cap_mlp_FRC': '83.3%', 'cap_control_featwide_R': '60.8%', '

## 14. H1–H8 verdicts (re-derived programmatically from the aggregates)

In [14]:
from neuroforge.evaluation.phase15_metrics import build_phase15_hypotheses
hyps = build_phase15_hypotheses(p15['aggregates'])
for h in sorted(hyps):
    print(f"{h}: {hyps[h]['status']}")
    print(f"    {hyps[h]['evidence']}")
assert set(hyps) == set(p15['hypotheses'])
assert all(hyps[h]['status'] == p15['hypotheses'][h]['status'] for h in hyps)
print('stored verdicts match fresh derivation: OK')

H1: SUPPORTED


    Depth depth3 vs depth-1 baseline on R: mean gain +4.7pp, positive on 3/3 seeds.
H2: SUPPORTED
    Aggregation agg_max vs mean baseline on R: mean gain +4.2pp, positive on 3/3 seeds.
H3: SUPPORTED
    MLP relational update vs linear update on R: mean gain +3.9pp, positive on 3/3 seeds. (+600 params, +576 FLOPs).
H4: NOT SUPPORTED
    Candidate destruction dependence delta -1.7pp on R; no evidence of topology-limited dependence.
H5: NOT SUPPORTED
    RC -0.3pp, FRC +0.6pp: isolated R gains did not transfer compositionally.
H6: PARTIALLY SUPPORTED
    Relational gain +1.7pp vs F/C gain +0.1pp (weak separation).
H7: NOT SUPPORTED
    Gap does not close (-2.5pp).
H8: NOT SUPPORTED
    Ceiling delta +0.5pp mixed mean.
stored verdicts match fresh derivation: OK


## 15. Final CASE classification (programmatic — no manually typed numbers)

In [15]:
from neuroforge.evaluation.phase15_metrics import select_phase15_case
case, label = select_phase15_case(hyps, p15['aggregates']['gap']['baseline_gap_mean'],
                                 p15['aggregates']['compositional'].get('R_gain_mean', 0.0))
print(f'Programmatic verdict: {case} — {label}')
assert case == p15['verdict_case'], 'stored case must equal fresh derivation'
print(f"Minimal validated intervention: {p15['minimal_intervention']['intervention']} ({p15['minimal_intervention']['outcome']})")
print()
print('Summary of evidence (all values loaded, none typed):')
print(f"  - baseline R: {p15['baseline_perf_mean']['R']*100:.1f}%, probe gap: {p15['aggregates']['gap']['baseline_gap_mean']*100:.1f}pp")
print(f"  - H1: {hyps['H1']['status']}, H2: {hyps['H2']['status']}, H3: {hyps['H3']['status']}, H4: {hyps['H4']['status']}")
print(f"  - H5: {hyps['H5']['status']}, H6: {hyps['H6']['status']}, H7: {hyps['H7']['status']}, H8: {hyps['H8']['status']}")

Programmatic verdict: CASE B — Additional relational propagation is sufficient
Minimal validated intervention: depth3 (Outcome A (depth))

Summary of evidence (all values loaded, none typed):
  - baseline R: 57.8%, probe gap: 19.2pp
  - H1: SUPPORTED, H2: SUPPORTED, H3: SUPPORTED, H4: NOT SUPPORTED
  - H5: NOT SUPPORTED, H6: PARTIALLY SUPPORTED, H7: NOT SUPPORTED, H8: NOT SUPPORTED
